# Agentic AI and Spatial Grounding

This notebook picks up where `TrafficSafety.ipynb` leaves off: it reads the
published Frankfurt traffic safety **risk layer**, retrieves the highest
confidence locations, and generates ranked, explainable recommendations
using `traffic_safety.recommendations`.

Set `RISK_LAYER_ITEM_ID` below to the item id returned by
`publish_hotspot_layer(...)` in the previous session.

## Problem Statement

A general-purpose language model has no inherent knowledge of a specific city's streets, and no access to statistically derived risk information. Asked directly where traffic safety should be prioritized, such a model would either decline to answer meaningfully or produce an answer without a verifiable basis.

**Spatial grounding** addresses this by supplying the model with structured, already-computed evidence about specific locations, and constraining its task to explaining that evidence rather than discovering or inventing it. The model's role is narrowed from "know everything about a place" to "produce a clear explanation of a fact pattern it has been given." This narrowing of scope is what makes the resulting explanation traceable and defensible: every statement in the output can be traced back to a value computed in the first session, rather than to the model's own assumptions.

### What Spatial Grounding Means

Spatial grounding provides an AI system with an understanding of:

- Location
- Proximity
- Accessibility
- Connectivity
- Spatial relationships
- Geographic context

A traditional language model can process descriptions of traffic safety issues in the abstract. A spatially grounded agent can reason about where risks occur, what surrounding factors are relevant, and which locations warrant priority for action — because it is given concrete evidence about those places rather than asked to infer it.

In [1]:
%load_ext dotenv

In [2]:
%dotenv

In [73]:
import os

import pandas as pd
from arcgis.gis import GIS
from arcgis.map import Map

from traffic_safety.recommendations import (
    TemplateLLMClient,
    get_risk_layer,
    get_top_risk_locations,
    rank_recommendations,
)

In [4]:
home_portal = GIS("home")
risk_layer_item_id = os.getenv("RISK_LAYER_ITEM_ID")
risk_layer = get_risk_layer(home_portal, risk_layer_item_id)

## Methodology

1. Publish the traffic safety risk layer (the outcome of the first session).
2. Expose the relevant maps, layers, and spatial analysis results to the agent.
3. Allow the agent to retrieve additional geographic context around the highest-priority locations.
4. Allow the agent to reason over locations and the relationships between them.
5. Generate explainable recommendations grounded in this evidence.
6. Present the recommendations, together with their supporting evidence, for human review.

This sequence allows an AI system to combine language-based reasoning with concrete geospatial knowledge, rather than relying solely on free-text description.

## Top risk locations

Query the risk layer for the locations with the highest, most persistent
confidence (`Gi_Bin_median`).

In [5]:
top_locations = get_top_risk_locations(risk_layer, top_n=5)
top_locations[["OBJECTID", "Gi_Bin_median", "Gi_Bin_years_observed", "Gi_Bin_min", "Gi_Bin_max"]]

,OBJECTID,Gi_Bin_median,Gi_Bin_years_observed,Gi_Bin_min,Gi_Bin_max
0,2,3.0,7,0.0,3.0
1,5,3.0,7,0.0,3.0
2,6,3.0,7,0.0,3.0
3,7,3.0,7,0.0,3.0
4,8,3.0,7,0.0,3.0


### Evidence Before Explanation

Each recommendation is derived from a structured evidence package describing, at minimum: a risk score reflecting persistence (carried over from the first session), the number of years that score was observed, the range of confidence values observed, and the amount of relevant surrounding context identified. The model is only asked to turn this evidence package into readable prose; it does not compute or select the evidence itself, and it is not asked to assess locations the evidence package does not already describe.

A key architectural decision deliberately separates **the mechanism that produces the explanatory text** from the rest of the workflow, so that different underlying models — whether a simple, deterministic placeholder, a hosted cloud model, or a locally run model — can be used interchangeably, without changing how evidence is gathered, structured, or reviewed. The specific generation mechanism shown in this session is chosen to keep the demonstration self-contained and reliable; richer generation approaches, including locally hosted models, are introduced in the following session using exactly the same evidence and the same overall workflow.

### On Supporting Geographic Context

The approach supports enriching each location's evidence package with nearby geographic context — for example, nearby infrastructure, land use, or points of interest — retrieved through a standard spatial intersection query. This capability exists as part of the overall approach. Depending on what supporting datasets are available and wired in for a given presentation, the context component of the evidence may or may not be populated; where it is not, this should be stated plainly rather than implied to be a limitation of the underlying method.

In [74]:
from arcgis.features import Feature, FeatureLayer, FeatureSet
import json

def get_context_layer(portal: GIS, item_id: str):
    """Retrieves the published traffic safety risk layer by item id.

    Args:
        portal (GIS): An authenticated GIS object.
        item_id (str): The item id of the hosted feature layer.

    Returns:
        FeatureLayer: The first layer of the context layer item.
    """
    item = portal.content.get(item_id)
    return item.layers[0]
    
def read_geographic_context():
    filepath = os.getenv("SCHOOLS_FILE")
    with open(filepath, 'r', encoding='utf-8') as file_in:
        schools_data = json.load(file_in)
        return FeatureSet.from_geojson(schools_data)

def show_context_map(portal: GIS, context_layer: FeatureLayer):
    basemap_item_id = os.getenv("CUSTOM_BASEMAP_ITEM_ID")
    if basemap_item_id:
        portal_item = portal.content.get(basemap_item_id)
        map_view = Map(portal_item)
    else:
        map_view = portal.map("Germany")
        map_view.basemap.basemap = "osm"
    map_view.content.add(context_layer)
    map_view.zoom_to_layer(context_layer)
    return map_view
    

#schools_fset = read_geographic_context()
#schools_fset.sdf
schools_layer_item_id = os.getenv("SCHOOLS_LAYER_ITEM_ID")
context_layer = get_context_layer(home_portal, schools_layer_item_id)
show_context_map(home_portal, context_layer)

Map()

In [80]:
# Buffer-based spatial grounding: find schools within a fixed walking-distance
# radius of each top risk location, then hand that buffered search area over
# to rank_recommendations so it drives the same live context query the
# generative step relies on.
from traffic_safety.recommendations import (
    get_buffer_context,
    persist_context_layer,
    load_context_layer,
    persist_context_geometries,
    load_context_geometries,
)

def generate_buffer_based_spatial_grounding(portal: GIS, top_locations: pd.DataFrame, context_layer: FeatureLayer):
    context_sdf = context_layer.query(out_fields="*", out_sr=4326).sdf
    output_dir = os.getenv("OUTPUT_DIRECTORY")

    context_geometries = {}
    all_matches = []
    for _, location in top_locations.iterrows():
        matches, detail = get_buffer_context(
            portal, context_sdf, location["SHAPE"], distance_meters=200,
        )
        print(f"Location {location['OBJECTID']}: {len(matches)} school(s) within {detail['buffer_meters']}m")
        context_geometries[location["OBJECTID"]] = detail["buffer_geometry"]
        matches["loc_id"] = location["OBJECTID"]
        all_matches.append(matches)
    
    # Combine every location's matched schools into a single GeoJSON file (a
    # `loc_id` column keeps each match traceable to its risk location), plus one
    # file for the search areas themselves. This is what lets the audience
    # re-run recommendations later without recomputing buffers.
    combined_context = pd.concat(all_matches, ignore_index=True)
    combined_context.spatial.set_geometry("SHAPE")
    context_layer_path = persist_context_layer(combined_context, "context_buffer", persist_dir=output_dir)
    print(f"Persisted context layer to {context_layer_path}")
    
    context_geometries_path = persist_context_geometries(context_geometries, name="context_geometries_buffer", persist_dir=output_dir)
    print(f"Persisted context geometries to {context_geometries_path}")
    return combined_context, context_geometries

def load_buffer_based_context():
    output_dir = os.getenv("OUTPUT_DIRECTORY")
    context_layer_path = os.path.join(output_dir, "context_buffer.geojson")
    combined_context = load_context_layer(context_layer_path)
    context_geometries_path = os.path.join(output_dir, "context_geometries_buffer.geojson")
    context_geometries = load_context_geometries(context_geometries_path)
    return combined_context, context_geometries

def generate_grounding_renderer():
    return {
        "type": "simple",
        "symbol": {
            "type": "esriSFS",
            "style": "esriSFSSolid",
            "color": [255, 182, 193, 204],
            "outline": {
                "color": [255, 0, 0, 255],
                "width": 1
            }
        }
    }

def show_grounding_map(portal: GIS, context_geometries: dict):
    basemap_item_id = os.getenv("CUSTOM_BASEMAP_ITEM_ID")
    if basemap_item_id:
        portal_item = portal.content.get(basemap_item_id)
        map_view = Map(portal_item)
    else:
        map_view = portal.map("Germany")
        map_view.basemap.basemap = "osm"
    drawing_info = { 
        "renderer": generate_grounding_renderer()
    }
    features = []
    for _, geometry in context_geometries.items():
        features.append(
            Feature(
                geometry=geometry,
                attributes={}
            )
        )
    fset = FeatureSet(features)
    map_view.content.add(fset, drawing_info=drawing_info, options={"opacity": 0.7})
    map_view.zoom_to_layer(fset)
    return map_view

In [51]:
combined_context, context_geometries = generate_buffer_based_spatial_grounding(home_portal, top_locations, context_layer)
combined_context

Location 2: 0 school(s) within 200m
Location 5: 1 school(s) within 200m
Location 6: 0 school(s) within 200m
Location 7: 1 school(s) within 200m
Location 8: 1 school(s) within 200m
Persisted context layer to /temp/DevSummit2026\context_buffer.geojson
Persisted context geometries to /temp/DevSummit2026\context_geometries_buffer.geojson


,F_id,amenity,check_date,name,wheelchair,addr_city,addr_housenumber,addr_postcode,addr_street,contact_email,...,note,isced,school,old_name,contact_facebook,contact_instagram,fax,ObjectId,SHAPE,loc_id
0,node/10002261849,school,2026-05-11,SRH Fachschule für Sozialpädagogik,<NA>,Frankfurt am Main,199,60326,Mainzer Landstraße,soziales.frankfurt.fs@srh.de,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,33,"{""x"": 8.6493335, ""y"": 50.10466840000001, ""spat...",5
1,node/11585362873,school,2026-03-26,Genussakademie Medienhaus,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,34,"{""x"": 8.6570742, ""y"": 50.1091567, ""spatialRefe...",7
2,node/11585362873,school,2026-03-26,Genussakademie Medienhaus,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,34,"{""x"": 8.6570742, ""y"": 50.1091567, ""spatialRefe...",8


In [81]:
combined_context, context_geometries = load_buffer_based_context()
show_grounding_map(home_portal, context_geometries)

Map()

### Alternative: service-area-based spatial grounding

Instead of a fixed-radius buffer, `get_service_area_context` measures
actual walking time along the street network via
`arcgis.network.analysis.generate_service_areas`. It mirrors
`get_buffer_context` exactly - same `(gis, context_sdf, geometry, ...)` call
shape, same `(matches, detail)` return with a search-area polygon in
`detail["service_area_geometry"]` - so it drops into the same per-location
loop and the same `rank_recommendations` re-query flow as the buffer cells
above. Run this cell as an alternative to the buffer cells, not in addition
to them.

`travel_mode` names are account-specific. Confirm "Walking Time" is
registered for your organization with `arcgis.network.analysis.get_travel_modes(gis)`
before relying on the default.

In [82]:
# Service-area-based spatial grounding: find schools within a 15-minute
# walking service area of each top risk location, then hand that service
# area polygon over to rank_recommendations so it drives the same live
# context query the generative step relies on.
from traffic_safety.recommendations import (
    get_service_area_context,
    persist_context_layer,
    load_context_layer,
    persist_context_geometries,
    load_context_geometries,
)

def generate_service_area_based_spatial_grounding(portal: GIS, top_locations: pd.DataFrame, context_layer: FeatureLayer):
    context_sdf = context_layer.query(out_fields="*", out_sr=4326).sdf
    output_dir = os.getenv("OUTPUT_DIRECTORY")

    context_geometries = {}
    all_matches = []
    for _, location in top_locations.iterrows():
        matches, detail = get_service_area_context(
            portal, context_sdf, location["SHAPE"],
            travel_mode="Walking Time", break_minutes=15,
        )
        print(f"Location {location['OBJECTID']}: {len(matches)} school(s) within {detail['break_minutes']} min ({detail['travel_mode']})")
        context_geometries[location["OBJECTID"]] = detail["service_area_geometry"]
        matches["loc_id"] = location["OBJECTID"]
        all_matches.append(matches)

    # Combine every location's matched schools into a single GeoJSON file (a
    # `loc_id` column keeps each match traceable to its risk location), plus one
    # file for the search areas themselves. This is what lets the audience
    # re-run recommendations later without recomputing service areas.
    combined_context = pd.concat(all_matches, ignore_index=True)
    combined_context.spatial.set_geometry("SHAPE")
    context_layer_path = persist_context_layer(combined_context, "context_service_area", persist_dir=output_dir)
    print(f"Persisted context layer to {context_layer_path}")

    context_geometries_path = persist_context_geometries(context_geometries, name="context_geometries_service_area", persist_dir=output_dir)
    print(f"Persisted context geometries to {context_geometries_path}")
    return combined_context, context_geometries

def load_service_area_based_context():
    output_dir = os.getenv("OUTPUT_DIRECTORY")
    context_layer_path = os.path.join(output_dir, "context_service_area.geojson")
    combined_context = load_context_layer(context_layer_path)
    context_geometries_path = os.path.join(output_dir, "context_geometries_service_area.geojson")
    context_geometries = load_context_geometries(context_geometries_path)
    return combined_context, context_geometries

In [83]:
combined_context_extended, context_geometries_extended = generate_service_area_based_spatial_grounding(home_portal, top_locations, context_layer)
#combined_context_extended

Location 2: 4 school(s) within 15 min (Walking Time)
Location 5: 7 school(s) within 15 min (Walking Time)
Location 6: 7 school(s) within 15 min (Walking Time)
Location 7: 7 school(s) within 15 min (Walking Time)
Location 8: 5 school(s) within 15 min (Walking Time)
Persisted context layer to /temp/DevSummit2026\context_service_area.geojson
Persisted context geometries to /temp/DevSummit2026\context_geometries_service_area.geojson


In [84]:
combined_context_extended, context_geometries_extended = load_service_area_based_context()
show_grounding_map(home_portal, context_geometries_extended)

Map()

## Generate explainable recommendations

`rank_recommendations` builds an evidence package per location and passes it
to an `LLMClient`. By default this uses the dependency-free
`TemplateLLMClient`, so the demo runs without any external AI service
configured. Swap in `traffic_safety.governance.OllamaLLMClient` (next
session) or any hosted-model client implementing the same `generate(prompt)`
interface to produce richer, free-form text from the exact same evidence.

In [12]:
recommendations = rank_recommendations(
    risk_layer,
    context_layer=context_layer,
    context_geometries=context_geometries,
    context_method="buffer",
    top_n=5,
    llm_client=TemplateLLMClient(),
)
for recommendation in recommendations:
    print(recommendation["recommendation"])
    print("-" * 80)

Location 2 shows a persistent risk score of 3.00 (range [0.0, 3.0]), observed across 7 year(s), with 0 relevant context feature(s) nearby. Recommend prioritizing this location for traffic safety review.
--------------------------------------------------------------------------------
Location 5 shows a persistent risk score of 3.00 (range [0.0, 3.0]), observed across 7 year(s), with 1 relevant context feature(s) nearby. Recommend prioritizing this location for traffic safety review.
--------------------------------------------------------------------------------
Location 6 shows a persistent risk score of 3.00 (range [0.0, 3.0]), observed across 7 year(s), with 0 relevant context feature(s) nearby. Recommend prioritizing this location for traffic safety review.
--------------------------------------------------------------------------------
Location 7 shows a persistent risk score of 3.00 (range [0.0, 3.0]), observed across 7 year(s), with 1 relevant context feature(s) nearby. Recommend

In [9]:
recommendations = rank_recommendations(
    risk_layer,
    context_layer=context_layer,
    context_geometries=context_geometries_extended,
    context_method="service_areas",
    top_n=5,
    llm_client=TemplateLLMClient(),
)
for recommendation in recommendations:
    print(recommendation["recommendation"])
    print("-" * 80)

NameError: name 'context_geometries_extended' is not defined

### Preview: the same evidence through a local model

`TemplateLLMClient` above exists so this session runs with zero external
dependencies - it renders the evidence package as plain text, with no model
involved. Nothing about the evidence package changes if the `LLMClient` is
swapped: `traffic_safety.governance.OllamaLLMClient` implements the exact
same `generate(prompt)` interface, backed by a locally running Ollama
model instead.

This cell reuses the first location's evidence (stripping out its
`TemplateLLMClient` text) and regenerates it with `OllamaLLMClient`, so the
two outputs can be compared side by side on the same underlying facts. The
full local-inference and human-review workflow is the subject of the next
session, `TrustedGeoAI.ipynb` - this is just a preview of the difference.

In [17]:
from IPython.display import display, Markdown
from traffic_safety.governance import OllamaLLMClient
from traffic_safety.recommendations import generate_recommendation


def generate_recommendations(recommendations: list[dict]):
    for recommendation in recommendations:
        sample_evidence = {key: value for key, value in recommendation.items() if key != "recommendation"}

        display(Markdown("**TemplateLLMClient (deterministic, no model):**"))
        display(Markdown(recommendation["recommendation"]))
        display(Markdown("-" * 80))
    
        try:
            ollama_recommendation = generate_recommendation(sample_evidence, llm_client=OllamaLLMClient())
            display(Markdown("**OllamaLLMClient (local model, same evidence):**"))
            display(Markdown(ollama_recommendation["recommendation"]))
        except Exception as error:
            print(f"OllamaLLMClient unavailable ({error}). Start a local Ollama server to see this comparison.")
            return

In [18]:
generate_recommendations(recommendations)

**TemplateLLMClient (deterministic, no model):**

Location 2 shows a persistent risk score of 3.00 (range [0.0, 3.0]), observed across 7 year(s), with 0 relevant context feature(s) nearby. Recommend prioritizing this location for traffic safety review.

--------------------------------------------------------------------------------

**OllamaLLMClient (local model, same evidence):**

**Recommendation: Prioritize Location 2 for Traffic Safety Review**

**Risk Score Breakdown:**

* Persistent Risk Score: 3.00
* Risk Score Range: [0.0, 3.0]
* Duration of Observations: 7 years

**Context Features:**

* 0 relevant context features nearby

**Prioritization Rationale:**

The persistent risk score of 3.00 at Location 2 suggests a moderate to high risk of traffic-related incidents or hazards. Given that the risk score is observed across 7 years, it indicates a consistent and potentially ongoing issue.

The lack of relevant context features nearby reduces the potential for external influences on the risk score, making it more likely that the issue is intrinsic to the location itself.

**Recommendation:**

Due to the persistent and moderate risk score, Location 2 should be prioritized for a thorough traffic safety review. This review should aim to identify the root causes of the issue and recommend mitigating measures to reduce the risk of traffic-related incidents or hazards.

**Action Items:**

1. Conduct a detailed site visit to Location 2 to gather more information about the context features and any potential contributing factors to the risk score.
2. Review relevant data and statistics to better understand the risk score's persistence and potential trends.
3. Consult with experts and stakeholders to identify potential solutions and recommend mitigation measures.
4. Develop and implement a plan to address the identified risks and ensure the traffic safety of the location.

...

**TemplateLLMClient (deterministic, no model):**

Location 8 shows a persistent risk score of 3.00 (range [0.0, 3.0]), observed across 7 year(s), with 1 relevant context feature(s) nearby. Recommend prioritizing this location for traffic safety review.

--------------------------------------------------------------------------------

**OllamaLLMClient (local model, same evidence):**

**Location 8 Risk Score Analysis**

Based on the provided information, Location 8 exhibits a persistent risk score of 3.00, which falls within the moderate to high risk category. The risk score range of [0.0, 3.0] indicates a relatively high level of risk, suggesting that the location is prone to accidents or hazardous conditions.

**Key Factors Contributing to the Risk Score:**

1. **Persistent Risk Score:** The risk score has been consistent over 7 years, indicating that the issue persists and may be a recurring problem.
2. **Relevant Context Features:** The presence of 1 relevant context feature(s) nearby suggests that the location is part of a larger network or system that may contribute to the risk.

**Recommendations:**

Given the elevated risk score and persistent nature of the issue, I strongly recommend prioritizing Location 8 for a thorough traffic safety review. This review should aim to identify the root causes of the risk and develop effective mitigation strategies to reduce the likelihood of accidents or hazardous conditions.

**Prioritization Rationale:**

1. **Safety Concerns:** The moderate to high risk score indicates a potential safety concern that requires immediate attention.
2. **Persistent Issue:** The consistent risk score over 7 years suggests that the issue is not isolated and may be a recurring problem that requires ongoing attention.
3. **Preventative Measures:** By prioritizing Location 8 for a traffic safety review, we can proactively identify and address potential safety concerns, reducing the likelihood of accidents and improving overall safety.

**Next Steps:**

1. Conduct a comprehensive risk assessment to identify the root causes of the risk.
2. Develop and implement effective mitigation strategies to reduce the risk.
3. Monitor the location's risk score over time to ensure the implemented measures are effective.

By prioritizing Location 8 for a traffic safety review, we can help mitigate the risk and ensure a safer environment for all users.

## Next steps

Each `recommendation` dict is a self-contained evidence package ready for
human review. Continue to `TrustedGeoAI.ipynb` to see these recommendations
go through a human-in-the-loop approval workflow with a full audit trail,
and to swap in a local/sovereign model via `OllamaLLMClient`.